# Language Identification - Lab 02:  GPU Run



## 1. Environment Check & Imports



In [1]:
import os
import re
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt

try:
    import torch
except ImportError as exc:
    raise ImportError(
        "PyTorch is not installed in this kernel. In VS Code select the torch_env kernel, "
        "or run: conda activate torch_env && pip install torch"
    ) from exc

try:
    from datasets import load_dataset
except ImportError as exc:
    raise ImportError(
        "datasets is not installed in this kernel. Run: conda activate torch_env && pip install datasets"
    ) from exc

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DATA_DIR = Path.cwd()
TRAIN_CSV = DATA_DIR / "train.csv"
VAL_CSV = DATA_DIR / "val.csv"
TEST_CSV = DATA_DIR / "test.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python kernel is using:", os.sys.executable)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
else:
    print("CUDA is not available in this kernel; training will run on CPU.")

Python kernel is using: c:\anaconda3\envs\torch_env\python.exe
PyTorch: 2.5.1
Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA runtime: 12.1


## 2. Build or Load Lab 01 Dataset

If the CSV files exist, this section loads them. If they are missing, it streams IndicCorpV2 and creates the same 80/10/10 splits used in Lab 01.

In [2]:
LANGUAGES = {
    "asm_Beng": "Assamese",
    "ben_Beng": "Bengali",
    "brx_Deva": "Bodo",
    "doi_Deva": "Dogri",
    "gom_Deva": "Konkani",
    "guj_Gujr": "Gujarati",
    "hin_Deva": "Hindi",
    "kan_Knda": "Kannada",
    "kas_Arab": "Kashmiri",
    "mai_Deva": "Maithili",
    "mal_Mlym": "Malayalam",
    "mar_Deva": "Marathi",
    "mni_Mtei": "Manipuri (Meitei)",
    "npi_Deva": "Nepali",
    "ory_Orya": "Odia",
    "pan_Guru": "Punjabi",
    "san_Deva": "Sanskrit",
    "snd_Deva": "Sindhi",
    "tam_Taml": "Tamil",
    "tel_Telu": "Telugu",
    "urd_Arab": "Urdu",
    "khasi": "Khasi",
    "santhali": "Santali",
}

SENTENCES_PER_LANG = 1000
MAX_DOCS_TO_SCAN = 200_000
MIN_SENT_LEN = 15
MAX_SENT_LEN = 400

SENT_SPLIT_RE = re.compile(r'(?<=[??.!?])\s+')


def split_sentences(text):
    text = str(text).strip().replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)
    parts = SENT_SPLIT_RE.split(text)
    return [p.strip() for p in parts if p.strip()]


def is_valid_sentence(sentence):
    return MIN_SENT_LEN <= len(sentence) <= MAX_SENT_LEN


def collect_sentences(lang_code):
    seen = set()
    sentences = []
    ds = load_dataset(
        "ai4bharat/IndicCorpV2",
        "indiccorp_v2",
        split=lang_code,
        streaming=True,
    )
    for i, row in enumerate(ds):
        if i >= MAX_DOCS_TO_SCAN or len(sentences) >= SENTENCES_PER_LANG:
            break
        text = row.get("text", "")
        if not text:
            continue
        for sentence in split_sentences(text):
            if is_valid_sentence(sentence) and sentence not in seen:
                seen.add(sentence)
                sentences.append(sentence)
                if len(sentences) >= SENTENCES_PER_LANG:
                    break
    random.shuffle(sentences)
    return sentences[:SENTENCES_PER_LANG]


def create_lab01_csvs():
    rows = []
    for code_, name in LANGUAGES.items():
        sentences = collect_sentences(code_)
        flag = "" if len(sentences) >= SENTENCES_PER_LANG else "  <-- short"
        print(f"{code_:10s} {name:20s} {len(sentences):5d} sentences{flag}")
        rows.extend({"text": sent, "label": code_} for sent in sentences)

    df = pd.DataFrame(rows)
    train_parts, val_parts, test_parts = [], [], []
    for code_ in LANGUAGES:
        sub = df[df.label == code_].sample(frac=1, random_state=42).reset_index(drop=True)
        n = len(sub)
        n_train = int(n * 0.8)
        n_val = int(n * 0.1)
        train_parts.append(sub.iloc[:n_train])
        val_parts.append(sub.iloc[n_train:n_train + n_val])
        test_parts.append(sub.iloc[n_train + n_val:])

    train = pd.concat(train_parts).sample(frac=1, random_state=42).reset_index(drop=True)
    val = pd.concat(val_parts).sample(frac=1, random_state=42).reset_index(drop=True)
    test = pd.concat(test_parts).sample(frac=1, random_state=42).reset_index(drop=True)

    train.to_csv(TRAIN_CSV, index=False, encoding="utf-8")
    val.to_csv(VAL_CSV, index=False, encoding="utf-8")
    test.to_csv(TEST_CSV, index=False, encoding="utf-8")
    return train, val, test


if TRAIN_CSV.exists() and VAL_CSV.exists() and TEST_CSV.exists():
    train_df = pd.read_csv(TRAIN_CSV)
    val_df = pd.read_csv(VAL_CSV)
    test_df = pd.read_csv(TEST_CSV)
    print("Loaded existing CSV files.")
else:
    print("CSV files not found. Creating train.csv, val.csv, and test.csv from IndicCorpV2...")
    train_df, val_df, test_df = create_lab01_csvs()

print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)
train_df.head()

CSV files not found. Creating train.csv, val.csv, and test.csv from IndicCorpV2...


asm_Beng   Assamese              1000 sentences
ben_Beng   Bengali               1000 sentences
brx_Deva   Bodo                  1000 sentences
doi_Deva   Dogri                 1000 sentences
gom_Deva   Konkani               1000 sentences
guj_Gujr   Gujarati              1000 sentences
hin_Deva   Hindi                 1000 sentences
kan_Knda   Kannada               1000 sentences
kas_Arab   Kashmiri              1000 sentences
mai_Deva   Maithili              1000 sentences
mal_Mlym   Malayalam             1000 sentences
mar_Deva   Marathi               1000 sentences
mni_Mtei   Manipuri (Meitei)     1000 sentences
npi_Deva   Nepali                1000 sentences
ory_Orya   Odia                  1000 sentences
pan_Guru   Punjabi               1000 sentences
san_Deva   Sanskrit              1000 sentences
snd_Deva   Sindhi                1000 sentences
tam_Taml   Tamil                 1000 sentences
tel_Telu   Telugu                1000 sentences
urd_Arab   Urdu                  1000 se

,text,label
0,प्रकृतेः परतो नित्यमिन्द्रियैरप्यगोचरः । 012a,san_Deva
1,सुलुङाव मिथिनो मोननाय बादिब्ला जोनोम जानायनि उ...,brx_Deva
2,അസീം മാലിക്കിനെ സൂപ്രണ്ട് സ്ഥാനത്ത് നിന്ന് ഒഴി...,mal_Mlym
3,পৰৱৰ্তী সময়ত সমীপতে থকা আন এটা পৰিয়ালৰ ঘৰত ডকা...,asm_Beng
4,येथे नोकरदार माणसांचा भरणा आहे.,mar_Deva


## 3. Labels

In [3]:
label_list = sorted(train_df["label"].unique().tolist())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

y_train = train_df["label"].map(label2id).values.astype(np.int64)
y_val = val_df["label"].map(label2id).values.astype(np.int64)
y_test = test_df["label"].map(label2id).values.astype(np.int64)
n_classes = len(label_list)

print(f"Total target classes: {n_classes}")

Total target classes: 23


## 4. Feature Extraction

In [4]:
def word_tokenize(text):
    return re.findall(r'\w+', str(text), re.UNICODE)


def word_ngrams(tokens, n):
    return [' '.join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def char_ngrams(text, n):
    clean_text = str(text).replace(' ', '')
    return [clean_text[i:i + n] for i in range(len(clean_text) - n + 1)]


def extract_features(text):
    tokens = word_tokenize(text)
    feats = []
    feats += ['W1_' + t for t in word_ngrams(tokens, 1)]
    feats += ['W2_' + t for t in word_ngrams(tokens, 2)]
    feats += ['C2_' + t for t in char_ngrams(text, 2)]
    feats += ['C3_' + t for t in char_ngrams(text, 3)]
    feats += ['C4_' + t for t in char_ngrams(text, 4)]
    return feats

extract_features("?? ?? ??????? ????? ??")[:10]

['C2_??',
 'C2_??',
 'C2_??',
 'C2_??',
 'C2_??',
 'C2_??',
 'C2_??',
 'C2_??',
 'C2_??',
 'C2_??']

## 5. Vocabulary and IDF Variants

In [5]:
MAX_VOCAB_PER_TYPE = 8000
MIN_DOC_FREQ = 2


def build_vocab(texts):
    doc_freq = Counter()
    for text in texts:
        doc_freq.update(set(extract_features(text)))

    by_type = {}
    for feat, df in doc_freq.items():
        if df < MIN_DOC_FREQ:
            continue
        ftype = feat.split('_', 1)[0]
        by_type.setdefault(ftype, []).append((feat, df))

    vocab = {}
    idx = 0
    for ftype, items in by_type.items():
        items.sort(key=lambda x: -x[1])
        for feat, _ in items[:MAX_VOCAB_PER_TYPE]:
            vocab[feat] = idx
            idx += 1
    return vocab, doc_freq


vocab, doc_freq = build_vocab(train_df["text"].tolist())
n_docs = len(train_df)

idf_raw = np.zeros(len(vocab), dtype=np.float32)
for feat, idx in vocab.items():
    idf_raw[idx] = np.log(n_docs / (1 + doc_freq[feat]))

max_idf = float(np.max(idf_raw)) if len(idf_raw) else 1.0
idf_norm = idf_raw / max_idf if max_idf != 0 else idf_raw

print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 40000


## 6. Vectorization USing the TF-IDF 

In [6]:
def vectorize_custom(texts, vocab, idf_vec, tf_type="raw"):
    rows, cols, data = [], [], []
    for r, text in enumerate(texts):
        feats = extract_features(text)
        if not feats:
            continue
        tf_counts = Counter(feats)
        total_terms = sum(tf_counts.values())
        max_freq = max(tf_counts.values()) if tf_counts else 1

        for feat, count in tf_counts.items():
            c = vocab.get(feat)
            if c is None:
                continue
            if tf_type == "raw":
                tf_val = count
            elif tf_type == "norm_total":
                tf_val = count / total_terms
            elif tf_type == "norm_max":
                tf_val = 0.5 + 0.5 * (count / max_freq)
            else:
                raise ValueError(f"Invalid tf_type: {tf_type}")
            rows.append(r)
            cols.append(c)
            data.append(tf_val * idf_vec[c])

    X = csr_matrix((data, (rows, cols)), shape=(len(texts), len(vocab)), dtype=np.float32)
    norms = np.sqrt(np.asarray(X.multiply(X).sum(axis=1))).flatten()
    norms[norms == 0] = 1.0
    return csr_matrix(X.multiply(1.0 / norms[:, None]), dtype=np.float32)

## 7. GPU Softmax Logistic Regression

In [ ]:
class TorchSoftmaxRegression:
    # Initialize model weights, bias, learning rate, and regularization
    def __init__(self, n_features, n_classes, lr=0.5, reg=1e-5, device=DEVICE):
        self.W = torch.zeros((n_features, n_classes), dtype=torch.float32, device=device)
        self.b = torch.zeros(n_classes, dtype=torch.float32, device=device)
        self.lr = lr
        self.reg = reg
        self.device = device

    # Convert sparse NumPy batch into a PyTorch tensor
    def _batch_tensor(self, X_batch):
        return torch.from_numpy(
            X_batch.toarray().astype(np.float32, copy=False)
        ).to(self.device)

    # Predict classes for one batch
    def predict_batch(self, X_batch):
        X_t = self._batch_tensor(X_batch)
        scores = X_t @ self.W + self.b
        return torch.argmax(scores, dim=1).detach().cpu().numpy()

    # Predict classes for the complete dataset in batches
    def predict(self, X, batch_size=1024):
        preds = []
        for start in range(0, X.shape[0], batch_size):
            preds.append(self.predict_batch(X[start:start + batch_size]))
        return np.concatenate(preds) if preds else np.array([], dtype=np.int64)

    # Train the model using mini-batch gradient descent
    def fit(self, X, y, X_val, y_val, epochs=20, batch_size=256):
        n_samples = X.shape[0]
        y_np = np.asarray(y, dtype=np.int64)

        # Train for multiple epochs
        for epoch in range(epochs):
            perm = np.random.permutation(n_samples)
            total_loss = 0.0
            batches = 0

            # Process training data batch by batch
            for start in range(0, n_samples, batch_size):
                idx = perm[start:start + batch_size]
                X_batch = self._batch_tensor(X[idx])
                y_batch = torch.from_numpy(y_np[idx]).to(self.device)

                # Calculate class scores
                scores = X_batch @ self.W + self.b

                # Convert scores into class probabilities
                probs = torch.softmax(scores, dim=1)

                # Calculate cross-entropy loss with L2 regularization
                loss = torch.nn.functional.cross_entropy(
                    scores, y_batch
                ) + 0.5 * self.reg * torch.sum(self.W * self.W)

                # Convert true labels into one-hot vectors
                one_hot = torch.nn.functional.one_hot(
                    y_batch, num_classes=self.W.shape[1]
                ).float()

                # Calculate gradients of scores, weights, and bias
                grad_scores = (probs - one_hot) / X_batch.shape[0]
                grad_W = X_batch.T @ grad_scores + self.reg * self.W
                grad_b = grad_scores.sum(dim=0)

                # Update weights and bias using gradient descent
                self.W -= self.lr * grad_W
                self.b -= self.lr * grad_b

                # Accumulate batch loss
                total_loss += float(loss.detach().cpu())
                batches += 1

            # Evaluate the model on validation data
            val_preds = self.predict(X_val)
            val_acc = (val_preds == y_val).mean()

            # Display epoch training progress
            print(
                f"epoch {epoch + 1:02d} "
                f"loss={total_loss / max(batches, 1):.4f} "
                f"val_acc={val_acc:.4f}"
            )

        # Return final validation accuracy
        return float(val_acc)

## 8. Evaluation Metrics

In [8]:
def compute_macro_f1(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1

    f1s = []
    for i in range(n_classes):
        tp = cm[i][i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        f1s.append(f1)
    return float(np.mean(f1s)), cm

## 9. Run All 6 TF-IDF Experiments

In [9]:
EPOCHS = 20
BATCH_SIZE = 256
LR = 0.5
L2_REG = 1e-5

combinations = [
    ("Combo 1: Unnorm TF + Unnorm IDF", "raw", idf_raw),
    ("Combo 2: Unnorm TF + Norm IDF", "raw", idf_norm),
    ("Combo 3: Norm TF (Total Words) + Unnorm IDF", "norm_total", idf_raw),
    ("Combo 4: Norm TF (Total Words) + Norm IDF", "norm_total", idf_norm),
    ("Combo 5: Norm TF (Max Freq) + Unnorm IDF", "norm_max", idf_raw),
    ("Combo 6: Norm TF (Max Freq) + Norm IDF", "norm_max", idf_norm),
]

results_summary = []
models = {}

for name, tf_type, idf_vec in combinations:
    print("\n" + "=" * 80)
    print(f"Training: {name}")
    X_tr = vectorize_custom(train_df["text"].tolist(), vocab, idf_vec, tf_type)
    X_v = vectorize_custom(val_df["text"].tolist(), vocab, idf_vec, tf_type)
    X_te = vectorize_custom(test_df["text"].tolist(), vocab, idf_vec, tf_type)

    clf = TorchSoftmaxRegression(
        n_features=X_tr.shape[1],
        n_classes=n_classes,
        lr=LR,
        reg=L2_REG,
        device=DEVICE,
    )
    val_acc = clf.fit(X_tr, y_train, X_v, y_val, epochs=EPOCHS, batch_size=BATCH_SIZE)
    test_preds = clf.predict(X_te)
    test_f1, _ = compute_macro_f1(y_test, test_preds, n_classes)

    results_summary.append({
        "Combination": name,
        "Val Accuracy": val_acc,
        "Test Macro-F1": test_f1,
    })
    models[name] = clf

results_df = pd.DataFrame(results_summary).sort_values("Test Macro-F1", ascending=False)
results_df


Training: Combo 1: Unnorm TF + Unnorm IDF
epoch 01 loss=3.0473 val_acc=0.8691
epoch 02 loss=2.8689 val_acc=0.8857
epoch 03 loss=2.6970 val_acc=0.9017
epoch 04 loss=2.5329 val_acc=0.8896
epoch 05 loss=2.3782 val_acc=0.8948
epoch 06 loss=2.2335 val_acc=0.8913
epoch 07 loss=2.0997 val_acc=0.8965
epoch 08 loss=1.9768 val_acc=0.9048
epoch 09 loss=1.8645 val_acc=0.9078
epoch 10 loss=1.7630 val_acc=0.9087
epoch 11 loss=1.6714 val_acc=0.9122
epoch 12 loss=1.5883 val_acc=0.9187
epoch 13 loss=1.5133 val_acc=0.9165
epoch 14 loss=1.4457 val_acc=0.9200
epoch 15 loss=1.3844 val_acc=0.9200
epoch 16 loss=1.3286 val_acc=0.9196
epoch 17 loss=1.2781 val_acc=0.9213
epoch 18 loss=1.2318 val_acc=0.9222
epoch 19 loss=1.1893 val_acc=0.9222
epoch 20 loss=1.1505 val_acc=0.9243

Training: Combo 2: Unnorm TF + Norm IDF
epoch 01 loss=3.0472 val_acc=0.7996
epoch 02 loss=2.8688 val_acc=0.8891
epoch 03 loss=2.6969 val_acc=0.8913
epoch 04 loss=2.5331 val_acc=0.8917
epoch 05 loss=2.3783 val_acc=0.8957
epoch 06 loss=2.

,Combination,Val Accuracy,Test Macro-F1
5,Combo 6: Norm TF (Max Freq) + Norm IDF,0.933043,0.919692
3,Combo 4: Norm TF (Total Words) + Norm IDF,0.924348,0.911908
4,Combo 5: Norm TF (Max Freq) + Unnorm IDF,0.927826,0.911411
1,Combo 2: Unnorm TF + Norm IDF,0.921739,0.909823
0,Combo 1: Unnorm TF + Unnorm IDF,0.924348,0.909351
2,Combo 3: Norm TF (Total Words) + Unnorm IDF,0.923043,0.909061


## 10. Performance Comparison

In [1]:
plt.figure(figsize=(10, 5))
plt.barh(results_df["Combination"], results_df["Test Macro-F1"], color="steelblue")
plt.xlabel("Macro F1 Score")
plt.title("Performance Comparison Across TF-IDF Normalization Strategies")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

## 11. Quick Prediction Check

In [12]:
best_row = results_df.iloc[0]
best_name = best_row["Combination"]
best_tf_type = next(tf for name, tf, _ in combinations if name == best_name)
best_idf = next(idf for name, _, idf in combinations if name == best_name)
best_model = models[best_name]


def predict_language(text):
    X = vectorize_custom([text], vocab, best_idf, best_tf_type)
    pred_id = best_model.predict(X, batch_size=1)[0]
    code_ = id2label[pred_id]
    print(f"Predicted: {LANGUAGES.get(code_, code_)} ({code_}) using {best_name}")

predict_language("यह एक परीक्षण वाक्य है।")

Predicted: Hindi (hin_Deva) using Combo 6: Norm TF (Max Freq) + Norm IDF
